In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('dft-road-casualty-statistics-collision-last-5-years.csv')

In [3]:

df1=df[[
    "latitude",
    "longitude",
    "date",
    "speed_limit",
    "road_type",
    "time",
    "first_road_class",
    "first_road_number",
    "junction_detail",
    "junction_control",
    "second_road_class",
    "second_road_number",
    "pedestrian_crossing",
    "road_surface_conditions",
    "weather_conditions",
    "light_conditions",
    "carriageway_hazards",
    "urban_or_rural_area"
]]

In [4]:
df1=df1.dropna()

In [5]:
df1['date']=pd.to_datetime(df1['date'],format="%d/%m/%Y")

In [6]:
import h3
res=8
df1['h3_cell']=df1.apply(lambda r:h3.latlng_to_cell(r['latitude'],r['longitude'],res),axis=1)

In [7]:
df1.columns

Index(['latitude', 'longitude', 'date', 'speed_limit', 'road_type', 'time',
       'first_road_class', 'first_road_number', 'junction_detail',
       'junction_control', 'second_road_class', 'second_road_number',
       'pedestrian_crossing', 'road_surface_conditions', 'weather_conditions',
       'light_conditions', 'carriageway_hazards', 'urban_or_rural_area',
       'h3_cell'],
      dtype='object')

In [8]:
from tqdm import tqdm 

In [9]:
cell_day=df1.groupby(['h3_cell','date']).size().reset_index(name='accidents_today')

In [10]:
cell_day

,h3_cell,date,accidents_today
0,8809a44713fffff,2023-03-17,1
1,8809a44c29fffff,2020-11-12,1
2,8809a44c97fffff,2024-05-04,1
3,8809a44ca1fffff,2020-10-25,1
4,8809a44ca1fffff,2023-06-10,1
...,...,...,...
496836,881976d88dfffff,2021-04-16,1
496837,881976d8c5fffff,2021-07-26,1
496838,881976d8ebfffff,2024-07-06,1
496839,881976d935fffff,2021-10-03,1


In [11]:
start=cell_day['date'].min()
end=cell_day['date'].max()

In [12]:
full_dates=pd.date_range(start,end,freq='D').date

In [13]:
N=len(full_dates)

In [14]:
date_to_idx={d:i for i,d in enumerate(full_dates)}

In [15]:
cells = cell_day['h3_cell'].unique()

In [16]:
window=7

In [17]:
rows_out=[]

In [18]:
cell_day['date'] = pd.to_datetime(cell_day['date']).dt.date
cell_groups = cell_day.groupby('h3_cell')

In [19]:
for cell, g in tqdm(cell_groups, total=len(cell_groups), desc="cells"):
    a=np.zeros(N,dtype=np.int32)
    for _,r in g.iterrows():
        idx=date_to_idx[r['date']]
        a[idx]=int(r['accidents_today'])
    c=np.empty(N+1,dtype=np.int32)
    c[0] = 0
    np.cumsum(a, out=c[1:])
    idx_i=np.arange(N)
    right_idx=np.minimum(idx_i+window+1,N)
    left_idx=idx_i+1
    future=c[right_idx]-c[left_idx]
    label = (future > 0).astype(np.int8)
    valid_until=N-window
    valid_idx=np.arange(0,valid_until)
    df_cell = pd.DataFrame({
        'h3_cell': cell,
        'date': [full_dates[i] for i in valid_idx],
        'future_accidents': future[valid_idx],
        'label': label[valid_idx],
        'accidents_today': a[valid_idx]
    })
    rows_out.append(df_cell)
label_df = pd.concat(rows_out, ignore_index=True)
print("Total rows in label_df:", len(label_df))
print("Label counts:\n", label_df['label'].value_counts())


cells: 100%|█████████████████████████████| 78990/78990 [02:00<00:00, 656.40it/s]


Total rows in label_df: 143761800
Label counts:
 label
0    140500645
1      3261155
Name: count, dtype: int64


In [20]:
label_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 143761800 entries, 0 to 143761799
Data columns (total 5 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   h3_cell           object
 1   date              object
 2   future_accidents  int32 
 3   label             int8  
 4   accidents_today   int32 
dtypes: int32(2), int8(1), object(2)
memory usage: 3.3+ GB


In [21]:
label_df.isnull().sum()

h3_cell             0
date                0
future_accidents    0
label               0
accidents_today     0
dtype: int64

In [22]:

print("label_df rows, cols:", label_df.shape)
print("df1 rows, cols:", df1.shape)


print("label_df mem (MB):", label_df.memory_usage(deep=True).sum()/1024**2)
print("df1 mem (MB):", df1.memory_usage(deep=True).sum()/1024**2)


needed = ['h3_cell','speed_limit','road_type','first_road_class','first_road_number',
          'junction_detail','junction_control','second_road_class','second_road_number',
          'pedestrian_crossing','road_surface_conditions','weather_conditions',
          'light_conditions','carriageway_hazards','urban_or_rural_area']
print("available needed cols:", [c for c in needed if c in df1.columns])

label_df rows, cols: (143761800, 5)
df1 rows, cols: (503410, 19)
label_df mem (MB): 15492.51893234253
df1 mem (MB): 125.78336715698242
available needed cols: ['h3_cell', 'speed_limit', 'road_type', 'first_road_class', 'first_road_number', 'junction_detail', 'junction_control', 'second_road_class', 'second_road_number', 'pedestrian_crossing', 'road_surface_conditions', 'weather_conditions', 'light_conditions', 'carriageway_hazards', 'urban_or_rural_area']


In [23]:
import pandas as pd, numpy as np, gc
from math import ceil
BATCH_SIZE = 5000
OUT_DIR = "train_chunks_parquet"  
agg_cols = [
    'h3_cell','speed_limit','road_type','first_road_class','first_road_number',
    'junction_detail','junction_control','second_road_class','second_road_number',
    'pedestrian_crossing','road_surface_conditions','weather_conditions',
    'light_conditions','carriageway_hazards','urban_or_rural_area'
]
cols = [c for c in agg_cols if c in df1.columns]

df_small = df1[cols].copy()

def mode_or_first(s):
    m = s.mode()
    return m.iloc[0] if not m.empty else s.iloc[0]

cell_attrs = df_small.groupby('h3_cell').agg(mode_or_first).reset_index()

for c in cell_attrs.select_dtypes(include=['object']).columns:
    cell_attrs[c] = cell_attrs[c].astype('category')

print("cell_attrs shape:", cell_attrs.shape)
print(cell_attrs.head())

maps = {}
for col in cell_attrs.columns:
    if col == 'h3_cell':
        continue
    maps[col] = cell_attrs.set_index('h3_cell')[col].to_dict()

import os
os.makedirs(OUT_DIR, exist_ok=True)

unique_cells = label_df['h3_cell'].unique()
n_batches = ceil(len(unique_cells) / BATCH_SIZE)
print(f"Unique cells: {len(unique_cells)}, batches: {n_batches}")

for i in range(0, len(unique_cells), BATCH_SIZE):
    batch_cells = unique_cells[i:i + BATCH_SIZE]
    chunk = label_df[label_df['h3_cell'].isin(batch_cells)].copy()

    for col, m in maps.items():
        chunk[col] = chunk['h3_cell'].map(m)

   

    part_path = os.path.join(OUT_DIR, f"part_{i//BATCH_SIZE:04d}.parquet")
    chunk.to_parquet(part_path, index=False)  # fast; adjust engine if needed

    print(f"Wrote {part_path} rows={len(chunk)}")
    del chunk
    gc.collect()

print("All batches written to", OUT_DIR)

cell_attrs shape: (78990, 15)
           h3_cell  speed_limit  road_type  first_road_class  \
0  8809a44713fffff           20          9                 6   
1  8809a44c29fffff           30          6                 4   
2  8809a44c97fffff           60          6                 4   
3  8809a44ca1fffff           60          6                 3   
4  8809a44ca3fffff           60          6                 4   

   first_road_number  junction_detail  junction_control  second_road_class  \
0                  0                0                -1                  0   
1               9074               16                 4                  6   
2               9074                0                -1                  6   
3                  0                0                -1                  0   
4               9073                0                -1                  0   

   second_road_number  pedestrian_crossing  road_surface_conditions  \
0                  -1                    0   